# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Setup connections
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# 1. Pull raw data
raw_df = con.execute(f"""
    SELECT
        f.client_hash_id, f.content_hash_id, f.gsc_clicks, f.gsc_impressions,
        f.ga4_total_engagement_sec, f.ga4_sessions, f.sessions_ai,
        c.word_count
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE AND c.is_published IS TRUE AND c.is_deleted IS FALSE
""").df()

# 2. Content-Level Aggregation
frame = raw_df.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
    ga4_sessions=("ga4_sessions", "sum"),
    sessions_ai=("sessions_ai", "sum"),
    word_count=("word_count", "max")
).reset_index()

# 3. Engineer Normalized Features & Absolute Target
# Target: Any AI traffic puts a page in the top 1%
frame["is_high_ai_spike"] = (frame["sessions_ai"] >= 1).astype(int)

frame["avg_engagement_sec"] = frame["ga4_total_engagement_sec"] / frame["ga4_sessions"].clip(lower=1.0)
frame["avg_engagement_sec"] = frame["avg_engagement_sec"].fillna(0.0)

frame["ctr_computed"] = frame["gsc_clicks"] / frame["gsc_impressions"].clip(lower=1.0)

frame["has_word_count"] = frame["word_count"].notnull().astype(int)
frame["word_count_log"] = np.log1p(frame["word_count"].fillna(0.0))

# Backlink features completely dropped to prevent client-level memorization
FEATURES = ["avg_engagement_sec", "ctr_computed", "word_count_log", "has_word_count"]
TARGET = "is_high_ai_spike"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

*   **`avg_engagement_sec`:** Measures content consumption quality (engagement time per session). Missing values (0 sessions) are filled with `0.0`. It is a historical trailing metric available prior to future AI spike predictions.
*   **`ctr_computed`:** Traditional search relevance (clicks/impressions). Sourced from Google Search Console, available with a standard pipeline delay.
*   **`word_count_log`:** Structural depth, transformed using `log1p` to compress extreme outliers typical in web content length. Missing values are filled with `0.0` but paired with `has_word_count = 0` so the model differentiates between "thin content" and "uncrawled content." Available upon publication.
*   **`backlinks_clean` & `has_backlinks`:** Traditional SEO authority metrics. Missing values are filled with `0.0` and paired with a binary flag (`has_backlinks`) to prevent the fill from acting as a hidden category signal. Available periodically via crawler updates.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

clean_frame = frame.dropna(subset=[TARGET])
X = clean_frame[FEATURES]
X_leaky = clean_frame[FEATURES + ["sessions_ai"]]
y = clean_frame[TARGET]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clean_frame['client_hash_id']))

# Scale honest features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

# Scale leaky features
scaler_leaky = StandardScaler()
X_train_leaky = scaler_leaky.fit_transform(X_leaky.iloc[train_idx])
X_test_leaky = scaler_leaky.transform(X_leaky.iloc[test_idx])

# Honest Model
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y.iloc[train_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], model.predict_proba(X_test_scaled)[:, 1])

# Leaky Model
model_leaky = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_leaky.fit(X_train_leaky, y.iloc[train_idx])
leaky_auc = roc_auc_score(y.iloc[test_idx], model_leaky.predict_proba(X_test_leaky)[:, 1])

print("=== Leakage Trap Test ===")
print(f"Honest Grouped AUC: {honest_auc:.4f}")
print(f"Leaky Grouped AUC (+sessions_ai): {leaky_auc:.4f}")

=== Leakage Trap Test ===
Honest Grouped AUC: 0.5403
Leaky Grouped AUC (+sessions_ai): 1.0000


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

*   **`sessions_ai`:** Deliberately excluded from the final feature set because it mathematically defines the target variable `is_high_ai_spike`. Including it causes immediate proxy leakage, driving the AUC artificially to 1.0000.
*   **`gsc_impressions` (Raw Sum):** Excluded because raw impression sums act as a proxy for site-wide mainstream visibility, biasing the model away from identifying niche content depth.
*   **`ga4_total_engagement_sec` (Raw Sum):** Excluded in favor of normalized `avg_engagement_sec` to prevent the model from confusing raw, low-quality traffic volume with high-quality user engagement patterns.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.